# Corrected, leakage-free pipeline — Pharmacy Desert Index (PDI)

Run every cell top to bottom. Before running, upload the same CSV you've been using (the cell below opens Colab's file picker, same as your original notebook).

This version restores **all five original models** (Decision Tree, Random Forest, XGBoost, SVM, KNN) plus two baselines (trivial majority-class, logistic regression), and **all the original figures** (correlation heatmap, per-model feature importance bar charts, model comparison bar charts, SHAP summary plot, PDI histogram) — rebuilt under the corrected, leakage-free pipeline so Table 2/3 and Figures 2/3/4 keep their original shape in the revised manuscript.

## What changed vs. the original notebook, and why

1. **Split first.** The original applied SMOTE, standardization, and correlation-based feature selection to the *entire* dataset before splitting. The reported validation/test sizes (967/138/277 = 1382) match the *post-SMOTE* total, not the 735 real counties — synthetic counties were mixed into validation and test. This notebook splits the real counties **first**; everything else is fit on the training fold only. *(Editorial leakage bullet; Reviewer CL #4)*
2. The one county with a missing target value is **dropped** instead of mode-imputed. *(CL #3)*
3. Class imbalance is handled via **sample weighting** instead of SMOTE (tested as competitive-to-better given only 44 real non-desert counties total); validation/test stay real and naturally imbalanced.
4. **Correlation analysis is computed on the training fold only**, and shown as a heatmap (Figure 2 equivalent) for interpretability — but it is *not* used as a hard pre-filter, since a train-only correlation screen at the original 0.2363 threshold leaves almost no features (most of the original signal was an artifact of computing correlation on the SMOTE-inflated data). All models below use the full retained feature set, minus `% Living in Desert` which is excluded categorically as a near-restatement of the target. *(Editorial #1; Reviewer CL #10)*
5. **Every model's hyperparameters are grid-searched by balanced accuracy**, not raw accuracy, so tuning isn't biased toward the majority class. *(Addresses Editorial #10 / CL #5's misleading-accuracy point at the tuning stage, not just the final report.)*
6. **Every model's decision threshold is tuned** via inner cross-validation (Youden's J) instead of left at the default 0.5 — this matters a lot given the class imbalance.
7. Two baselines are included for comparison: a trivial always-predict-majority-class rule, and logistic regression. *(Editorial #10; CL #5/#6)*
8. A **leave-one-state-out (LOSO)** check and a **100-fold repeated cross-validation** robustness check are included, since a single split is noisy with only 44 real minority-class counties. *(CL #6)*
9. The **best model is chosen by 50-fold repeated cross-validation (5-fold × 10 repeats) on train+validation combined**, not a single 73-county validation split, and is the only model evaluated on the test set, preserving the original train/validate/test protocol. (An earlier version of this notebook chose the winner from the single validation split — a real run of that version picked Random Forest with a single-split test AUC of 0.90, but a 100-fold robustness check of that exact model averaged only AUC 0.63, with zero of the 100 folds coming close to 0.90. That gap meant the "winner" was noise from a too-small validation set, not a genuinely better model — this cell fixes the selection step itself, not just the final report.)
10. SHAP and the PDI's percentile ranks are computed **only on the 735 real counties** (fixing a bug found in the original code where they were computed against a reference population that was 47% synthetic), using whichever SHAP explainer is appropriate for the winning model type.

Read the printed output carefully — it's organized to map directly onto revised Tables 2, 3, 5, 6, 7 and Figures 2, 3, 4.

## 0. Setup — install SHAP and upload your data

In [ ]:
# SHAP isn't preinstalled in Colab -- same as your original notebook
!pip install shap


In [ ]:
# Upload the CSV (same as your original notebook)
from google.colab import files
uploaded = files.upload()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, RepeatedStratifiedKFold, ParameterGrid
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, confusion_matrix, classification_report,
    roc_auc_score, average_precision_score, roc_curve
)
from scipy.stats import rankdata

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 1. Load data

Edit `CSV_PATH` below to match the filename you just uploaded.

In [ ]:
CSV_PATH = '2024 All county combined full dataset - Sheet1 (1).csv'  # <-- EDIT THIS if your filename differs
df = pd.read_csv(CSV_PATH)
df.head()


## 2. Ratio → per-person conversion

Same transformation as the original notebook — not a leakage issue, so unchanged.

In [ ]:
def extract_numeric_part(s):
    return float(s.split(':')[0]) if isinstance(s, str) else np.nan

def invert_ratio(v):
    return 1 / v if pd.notnull(v) and v != 0 else np.nan

for col, newcol in [
    ('Primary Care Physicians Ratio', 'Primary Care Physicians per Person'),
    ('Dentist Ratio', 'Dentists per Person'),
    ('Mental Health Provider Ratio', 'Mental Health Providers per Person'),
    ('Other Primary Care Provider Ratio', 'Other Primary Care Providers per Person'),
]:
    df[col] = df[col].apply(extract_numeric_part)
    df[newcol] = df[col].apply(invert_ratio)
df = df.drop(columns=['Primary Care Physicians Ratio', 'Dentist Ratio',
                       'Mental Health Provider Ratio', 'Other Primary Care Provider Ratio'])


## 3. FIX — drop the county with an unknown target

The original notebook mode-imputed this; Reviewer CL flagged that as inappropriate. We drop it instead.

In [ ]:
n_before = len(df)
df = df[df['Desert Y/N'].notna()].copy()
print(f"[FIX] Dropped {n_before - len(df)} row(s) with missing target label.")

df['Desert Y/N'] = df['Desert Y/N'].map({'Y': 1, 'N': 0})
df[['County', 'State']] = df['County, State'].str.split(', ', expand=True)


## 4. Drop columns >20% missing

Computed on the full dataset -- uses only missingness counts, not label information, so this is not the leakage the reviewers flagged.

In [ ]:
missing_frac = df.isnull().mean()
cols_to_drop = [c for c in missing_frac[missing_frac > 0.20].index if c != 'Desert Y/N']
print(f"[Editorial #2, expanded] Columns dropped for >20% missing ({len(cols_to_drop)}): {cols_to_drop}")
df = df.drop(columns=cols_to_drop)


## 5. FIX — split the real counties FIRST

The single most important fix. Everything from here on is fit on the training fold only.

In [ ]:
EXCLUDE_ALWAYS = ['Desert Y/N', 'County', 'State', 'County, State', '% Living in Desert']  # Editorial #1
feature_cols = [c for c in df.columns if c not in EXCLUDE_ALWAYS]

X_all = df[feature_cols + ['County', 'State']]
y_all = df['Desert Y/N']

X_train, X_temp, y_train, y_temp = train_test_split(
    X_all, y_all, test_size=0.3, random_state=RANDOM_STATE, stratify=y_all
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=2/3, random_state=RANDOM_STATE, stratify=y_temp
)
print(f"[FIX] Real-county split: train={len(X_train)}, val={len(X_val)}, test={len(X_test)}")
print("Train class balance:\n", y_train.value_counts())
print("Val class balance:\n", y_val.value_counts())
print("Test class balance:\n", y_test.value_counts())


## 6. FIX — impute using TRAIN-fold state means only

In [ ]:
train_state_means = X_train.groupby('State')[feature_cols].mean()
overall_train_mean = X_train[feature_cols].mean()

def apply_impute(X):
    X = X.copy()
    for state in X['State'].unique():
        mask = X['State'] == state
        fill_vals = train_state_means.loc[state] if state in train_state_means.index else overall_train_mean
        for c in feature_cols:
            fv = fill_vals[c]
            if pd.isna(fv):
                fv = overall_train_mean[c]
            X.loc[mask, c] = X.loc[mask, c].fillna(fv)
    X[feature_cols] = X[feature_cols].fillna(X[feature_cols].mean())
    return X

X_train_imp = apply_impute(X_train)
X_val_imp = apply_impute(X_val)
X_test_imp = apply_impute(X_test)
print("[FIX] Imputation complete (train-fold statistics only).")


## 7. Correlation analysis — Figure 2 equivalent

Computed on the **training fold only** (the original computed this on the full post-SMOTE dataset, which is part of the leakage issue). Shown for interpretability; **not** used as a hard feature-selection filter below — a train-only screen at the original 0.2363 threshold leaves almost no features, since most of the original correlation strength was an artifact of the SMOTE-inflated data. All models use the full retained feature set instead, letting each model's own regularization and (for the winner) SHAP determine importance.

In [ ]:
train_corr_df = X_train_imp[feature_cols].copy()
train_corr_df['Desert Y/N'] = y_train.values
corr_with_target = train_corr_df.corr()['Desert Y/N'].drop('Desert Y/N').sort_values()
print("Train-only correlation with target (sorted):")
print(corr_with_target)

plt.figure(figsize=(10, max(6, len(feature_cols) * 0.25)))
corr_with_target.sort_values().plot(kind='barh', color='steelblue')
plt.title('Train-only correlation with Desert Y/N (Figure 2 equivalent)')
plt.xlabel('Pearson correlation coefficient')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(14, 11))
sns.heatmap(train_corr_df.corr(), cmap='coolwarm', center=0, annot=False)
plt.title('Correlation heatmap (training fold only)')
plt.tight_layout()
plt.show()


## 8. Manual SMOTE helper (train fold only, optional)

Not used in the main run below (sample-weighting is used instead), but available if you want to compare SMOTE-on-train as an alternative. No `imblearn` needed.

In [ ]:
def manual_smote(X, y, k=5, random_state=RANDOM_STATE):
    rng = np.random.RandomState(random_state)
    X = np.asarray(X, dtype=float)
    y = np.asarray(y)
    classes, counts = np.unique(y, return_counts=True)
    majority_count = counts.max()
    X_new, y_new = [X], [y]
    for cls, cnt in zip(classes, counts):
        n_gen = majority_count - cnt
        if n_gen <= 0:
            continue
        X_cls = X[y == cls]
        k_eff = min(k, len(X_cls) - 1)
        if k_eff < 1:
            synth = X_cls[rng.randint(0, len(X_cls), size=n_gen)]
        else:
            nn = NearestNeighbors(n_neighbors=k_eff + 1).fit(X_cls)
            _, neighbors = nn.kneighbors(X_cls)
            synth = np.zeros((n_gen, X.shape[1]))
            for i in range(n_gen):
                base = rng.randint(0, len(X_cls))
                nbr = neighbors[base][1:][rng.randint(0, k_eff)]
                gap = rng.rand()
                synth[i] = X_cls[base] + gap * (X_cls[nbr] - X_cls[base])
        X_new.append(synth)
        y_new.append(np.full(n_gen, cls))
    return np.vstack(X_new), np.concatenate(y_new)


## 9. Scale (fit on train only)

In [ ]:
scaler = StandardScaler().fit(X_train_imp[feature_cols])
X_train_scaled = scaler.transform(X_train_imp[feature_cols])
X_val_scaled = scaler.transform(X_val_imp[feature_cols])
X_test_scaled = scaler.transform(X_test_imp[feature_cols])

y_train_arr = y_train.values
n_pos = (y_train_arr == 1).sum()
n_neg = (y_train_arr == 0).sum()
sample_weight_train = np.where(y_train_arr == 0, n_pos / n_neg, 1.0)  # upweight the real minority class


## 10. Shared model-fitting framework

Every model below (baselines and all five classifiers) goes through the same four steps, so the comparison in Table 2 is fair and consistent:
1. Manual grid search, scored by **balanced accuracy** via 5-fold CV on the training fold only (3-fold for SVM, to keep runtime reasonable).
2. Refit the best hyperparameters on the full training fold.
3. Tune the decision threshold via inner 3-fold CV (Youden's J) on the training fold only — no validation/test leakage.
4. Evaluate on validation at the tuned threshold: confusion matrix, balanced accuracy, sensitivity, specificity, PPV, NPV, ROC-AUC, PR-AUC.

In [ ]:
def youden_threshold(y_true, y_prob):
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    return thr[np.argmax(tpr - fpr)]

def tune_threshold_inner_cv(build_fn, X, y, sw, n_splits=3):
    y = np.asarray(y)
    inner = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros(len(y))
    for tr, te in inner.split(X, y):
        m = build_fn()
        try:
            m.fit(X[tr], y[tr], sample_weight=sw[tr])
        except TypeError:
            m.fit(X[tr], y[tr])
        oof[te] = m.predict_proba(X[te])[:, 1]
    return youden_threshold(y, oof)

def eval_at(y_true, prob, thr):
    pred = (prob >= thr).astype(int)
    cm = confusion_matrix(y_true, pred)
    tn, fp, fn, tp = cm.ravel()
    out = dict(
        threshold=thr,
        balanced_accuracy=balanced_accuracy_score(y_true, pred),
        accuracy=accuracy_score(y_true, pred),
        sensitivity=tp / (tp + fn) if (tp + fn) else np.nan,
        specificity=tn / (tn + fp) if (tn + fp) else np.nan,
        ppv=tp / (tp + fp) if (tp + fp) else np.nan,
        npv=tn / (tn + fn) if (tn + fn) else np.nan,
    )
    try:
        out['roc_auc'] = roc_auc_score(y_true, prob)
        out['pr_auc'] = average_precision_score(y_true, prob)
    except Exception:
        out['roc_auc'] = np.nan
        out['pr_auc'] = np.nan
    return out, cm

def print_eval(name, metrics, cm):
    print(f"\n=== {name} (VAL) ===")
    print("Confusion matrix [[TN,FP],[FN,TP]]:\n", cm)
    for k, v in metrics.items():
        print(f"  {k}: {v:.3f}" if isinstance(v, (int, float)) and not pd.isna(v) else f"  {k}: {v}")

def manual_grid_search_cv(build_fn, param_grid, X, y, sw, cv=5):
    best_score = -np.inf
    best_params = None
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=RANDOM_STATE)
    for params in ParameterGrid(param_grid):
        scores = []
        for tr, te in skf.split(X, y):
            m = build_fn(**params)
            try:
                m.fit(X[tr], y[tr], sample_weight=sw[tr])
            except TypeError:
                m.fit(X[tr], y[tr])
            pred = m.predict(X[te])
            scores.append(balanced_accuracy_score(y[te], pred))
        mean_score = np.mean(scores)
        if mean_score > best_score:
            best_score = mean_score
            best_params = params
    return best_params, best_score

def plot_feature_importance(name, importances, feature_names, top_n=20):
    s = pd.Series(importances, index=feature_names).sort_values(ascending=False).head(top_n)
    plt.figure(figsize=(12, 6))
    s.plot(kind='bar', color='skyblue')
    plt.title(f'Feature Importances -- {name}')
    plt.ylabel('Importance')
    plt.tight_layout()
    plt.show()

# results and fitted models get collected here for the comparison table + best-model selection
results = {}
fitted_models = {}
build_fns = {}


## 11. Baselines — trivial majority-class + logistic regression

*(Editorial #10; Reviewer CL #5/#6)*

In [ ]:
majority_class = int(pd.Series(y_train_arr).mode()[0])
trivial_pred_val = np.full(len(y_val), majority_class)
cm = confusion_matrix(y_val, trivial_pred_val)
tn, fp, fn, tp = cm.ravel()
results['Trivial (majority class)'] = dict(
    threshold=np.nan, balanced_accuracy=balanced_accuracy_score(y_val, trivial_pred_val),
    accuracy=accuracy_score(y_val, trivial_pred_val),
    sensitivity=tp / (tp + fn) if (tp + fn) else np.nan, specificity=tn / (tn + fp) if (tn + fp) else np.nan,
    ppv=tp / (tp + fp) if (tp + fp) else np.nan, npv=tn / (tn + fn) if (tn + fn) else np.nan,
    roc_auc=0.5, pr_auc=np.nan,
)
print_eval('Trivial majority-class baseline', results['Trivial (majority class)'], cm)


In [ ]:
lr_grid = {'C': [0.05, 0.1, 0.5, 1, 5]}
def build_lr(**p):
    return LogisticRegression(max_iter=3000, class_weight='balanced', random_state=RANDOM_STATE, **p)

lr_best_params, lr_cv_score = manual_grid_search_cv(build_lr, lr_grid, X_train_scaled, y_train_arr, sample_weight_train, cv=5)
print('Best Logistic Regression params:', lr_best_params, '| CV balanced accuracy:', round(lr_cv_score, 3))
build_fns['Logistic Regression'] = lambda: build_lr(**lr_best_params)
lr = build_fns['Logistic Regression']()
lr.fit(X_train_scaled, y_train_arr)
fitted_models['Logistic Regression'] = lr
thr = tune_threshold_inner_cv(build_fns['Logistic Regression'], X_train_scaled, y_train_arr, sample_weight_train)
prob_val = lr.predict_proba(X_val_scaled)[:, 1]
results['Logistic Regression'], cm = eval_at(y_val, prob_val, thr)
print_eval('Logistic Regression baseline', results['Logistic Regression'], cm)

plot_feature_importance('Logistic Regression', lr.coef_[0], feature_cols)


## 12. Decision Tree

In [ ]:
dt_grid = {'max_depth': [3, 5, 7, 10], 'min_samples_split': [2, 5, 10], 'min_samples_leaf': [1, 2, 4], 'criterion': ['gini', 'entropy']}
def build_dt(**p):
    return DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight='balanced', **p)

dt_best_params, dt_cv_score = manual_grid_search_cv(build_dt, dt_grid, X_train_scaled, y_train_arr, sample_weight_train, cv=5)
print('Best Decision Tree params:', dt_best_params, '| CV balanced accuracy:', round(dt_cv_score, 3))
build_fns['Decision Tree'] = lambda: build_dt(**dt_best_params)
dt = build_fns['Decision Tree']()
dt.fit(X_train_scaled, y_train_arr)
fitted_models['Decision Tree'] = dt
thr = tune_threshold_inner_cv(build_fns['Decision Tree'], X_train_scaled, y_train_arr, sample_weight_train)
prob_val = dt.predict_proba(X_val_scaled)[:, 1]
results['Decision Tree'], cm = eval_at(y_val, prob_val, thr)
print_eval('Decision Tree', results['Decision Tree'], cm)

plot_feature_importance('Decision Tree', dt.feature_importances_, feature_cols)


## 13. Random Forest

In [ ]:
rf_grid = {'n_estimators': [100, 200, 300], 'max_depth': [None, 10, 15, 20], 'min_samples_split': [2, 5], 'min_samples_leaf': [1, 2]}
def build_rf(**p):
    return RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced', **p)

rf_best_params, rf_cv_score = manual_grid_search_cv(build_rf, rf_grid, X_train_scaled, y_train_arr, sample_weight_train, cv=5)
print('Best Random Forest params:', rf_best_params, '| CV balanced accuracy:', round(rf_cv_score, 3))
build_fns['Random Forest'] = lambda: build_rf(**rf_best_params)
rf = build_fns['Random Forest']()
rf.fit(X_train_scaled, y_train_arr)
fitted_models['Random Forest'] = rf
thr = tune_threshold_inner_cv(build_fns['Random Forest'], X_train_scaled, y_train_arr, sample_weight_train)
prob_val = rf.predict_proba(X_val_scaled)[:, 1]
results['Random Forest'], cm = eval_at(y_val, prob_val, thr)
print_eval('Random Forest', results['Random Forest'], cm)

plot_feature_importance('Random Forest', rf.feature_importances_, feature_cols)


## 14. XGBoost

The grid is intentionally weighted toward **shallow, regularized** configurations (`max_depth` 2-5, modest learning rates, meaningful L2) — testing during development found that, given only ~35 real minority-class counties per training fold, an unconstrained deep XGBoost overfits and performs barely better than chance on real held-out data, while a shallow/regularized one is competitive with or better than the other models.

In [ ]:
from xgboost import XGBClassifier

xgb_grid = {'max_depth': [2, 3, 4, 5], 'n_estimators': [100, 150, 200], 'learning_rate': [0.05, 0.1], 'reg_lambda': [1.0, 2.0, 3.0]}
def build_xgb(**p):
    return XGBClassifier(subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', random_state=RANDOM_STATE, **p)

xgb_best_params, xgb_cv_score = manual_grid_search_cv(build_xgb, xgb_grid, X_train_scaled, y_train_arr, sample_weight_train, cv=5)
print('Best XGBoost params:', xgb_best_params, '| CV balanced accuracy:', round(xgb_cv_score, 3))
build_fns['XGBoost'] = lambda: build_xgb(**xgb_best_params)
xgb_model = build_fns['XGBoost']()
xgb_model.fit(X_train_scaled, y_train_arr, sample_weight=sample_weight_train)
fitted_models['XGBoost'] = xgb_model
thr = tune_threshold_inner_cv(build_fns['XGBoost'], X_train_scaled, y_train_arr, sample_weight_train)
prob_val = xgb_model.predict_proba(X_val_scaled)[:, 1]
results['XGBoost'], cm = eval_at(y_val, prob_val, thr)
print_eval('XGBoost', results['XGBoost'], cm)

plot_feature_importance('XGBoost', xgb_model.feature_importances_, feature_cols)


## 15. Support Vector Machine

`probability=True` is required so we can tune a threshold and compute ROC-AUC/PR-AUC consistently with the other models (the original notebook didn't need this since it only used `.predict()`).

In [ ]:
svm_grid = {'C': [0.1, 1, 10], 'kernel': ['linear', 'rbf', 'poly'], 'gamma': ['scale', 'auto']}
def build_svm(**p):
    return SVC(probability=True, random_state=RANDOM_STATE, class_weight='balanced', **p)

svm_best_params, svm_cv_score = manual_grid_search_cv(build_svm, svm_grid, X_train_scaled, y_train_arr, sample_weight_train, cv=3)
print('Best SVM params:', svm_best_params, '| CV balanced accuracy:', round(svm_cv_score, 3))
build_fns['SVM'] = lambda: build_svm(**svm_best_params)
svm_model = build_fns['SVM']()
svm_model.fit(X_train_scaled, y_train_arr)
fitted_models['SVM'] = svm_model
thr = tune_threshold_inner_cv(build_fns['SVM'], X_train_scaled, y_train_arr, sample_weight_train, n_splits=3)
prob_val = svm_model.predict_proba(X_val_scaled)[:, 1]
results['SVM'], cm = eval_at(y_val, prob_val, thr)
print_eval('SVM', results['SVM'], cm)

if svm_model.kernel == 'linear':
    plot_feature_importance('SVM (linear)', svm_model.coef_[0], feature_cols)
else:
    print(f"Best SVM kernel is '{svm_model.kernel}' (non-linear) -- coefficient-based feature importance doesn't apply.")


## 16. K-Nearest Neighbors

KNN has no native feature-importance measure (same as the original notebook -- no feature importance section was produced for KNN there either).

In [ ]:
knn_grid = {'n_neighbors': [3, 5, 7, 9, 11], 'weights': ['uniform', 'distance'], 'metric': ['euclidean', 'manhattan', 'minkowski']}
def build_knn(**p):
    return KNeighborsClassifier(**p)

knn_best_params, knn_cv_score = manual_grid_search_cv(build_knn, knn_grid, X_train_scaled, y_train_arr, sample_weight_train, cv=5)
print('Best KNN params:', knn_best_params, '| CV balanced accuracy:', round(knn_cv_score, 3))
build_fns['KNN'] = lambda: build_knn(**knn_best_params)
knn_model = build_fns['KNN']()
knn_model.fit(X_train_scaled, y_train_arr)
fitted_models['KNN'] = knn_model
thr = tune_threshold_inner_cv(build_fns['KNN'], X_train_scaled, y_train_arr, sample_weight_train)
prob_val = knn_model.predict_proba(X_val_scaled)[:, 1]
results['KNN'], cm = eval_at(y_val, prob_val, thr)
print_eval('KNN', results['KNN'], cm)


## 17. Model comparison — Table 2 equivalent

Built programmatically from the `results` dict above (the original notebook hard-coded these numbers by hand from earlier cell outputs — this version reads them directly from the fitted models, so the table can't drift out of sync with the actual run).

In [ ]:
comparison_df = pd.DataFrame(results).T
comparison_df = comparison_df[['threshold', 'balanced_accuracy', 'accuracy', 'sensitivity', 'specificity', 'ppv', 'npv', 'roc_auc', 'pr_auc']]
comparison_df = comparison_df.sort_values('balanced_accuracy', ascending=False)
print("=== Model comparison on VALIDATION (real, imbalanced counties) -- Table 2 equivalent ===")
comparison_df.round(3)


In [ ]:
plt.figure(figsize=(10, 6))
comparison_df['balanced_accuracy'].sort_values().plot(kind='barh', color='steelblue')
plt.title('Model Comparison: Validation Balanced Accuracy')
plt.xlabel('Balanced Accuracy')
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
comparison_df[['sensitivity', 'specificity']].plot(kind='bar')
plt.title('Model Comparison: Sensitivity vs. Specificity (VAL)')
plt.ylabel('Score')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## 17b. Robust model comparison — repeated cross-validation (this is what picks "best model")

The table above is a **single 73-county validation split**, with only ~4 real non-desert counties in it — exactly the kind of small-sample estimate Reviewer CL warned about, and noisy enough to pick the wrong "best" model by chance alone. (This is not hypothetical: in an earlier run of this pipeline, the single-split winner scored AUC 0.90 on the held-out test set, but a 100-fold repeated-CV check of that same model — same hyperparameters, same threshold logic — averaged only AUC 0.63, with *zero* of the 100 CV folds reaching anywhere near the single-split number. That gap means the single-split "winner" was noise, not a genuinely better model.)

This cell re-scores the five already-tuned classifiers (same hyperparameters chosen above) using **repeated stratified 5-fold cross-validation on the combined train+validation counties** (50 folds total: 5-fold × 10 repeats) instead of one split. **The model selected as "best" below is chosen from this table, not the single-split table above.** *(Directly addresses Reviewer CL #6's small-sample-instability concern, applied to model selection itself, not just the final reported metric.)*

In [ ]:
X_trainval_imp = apply_impute(pd.concat([X_train, X_val]))
y_trainval = pd.concat([y_train, y_val])
scaler_cv = StandardScaler().fit(X_trainval_imp[feature_cols])
X_trainval_scaled = scaler_cv.transform(X_trainval_imp[feature_cols])
y_trainval_arr = y_trainval.values
sw_trainval = np.where(y_trainval_arr == 0, (y_trainval_arr == 1).sum() / max((y_trainval_arr == 0).sum(), 1), 1.0)

cv_candidates = ['Decision Tree', 'Random Forest', 'XGBoost', 'SVM', 'KNN']
rskf_sel = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=RANDOM_STATE)
cv_robust_results = {}
for name in cv_candidates:
    build_fn = build_fns[name]
    fold_aucs, fold_bas = [], []
    for tr_idx, te_idx in rskf_sel.split(X_trainval_scaled, y_trainval_arr):
        m = build_fn()
        try:
            m.fit(X_trainval_scaled[tr_idx], y_trainval_arr[tr_idx], sample_weight=sw_trainval[tr_idx])
        except TypeError:
            m.fit(X_trainval_scaled[tr_idx], y_trainval_arr[tr_idx])
        prob = m.predict_proba(X_trainval_scaled[te_idx])[:, 1]
        fold_aucs.append(roc_auc_score(y_trainval_arr[te_idx], prob))
        fold_bas.append(balanced_accuracy_score(y_trainval_arr[te_idx], (prob >= 0.5).astype(int)))
    cv_robust_results[name] = dict(
        auc_mean=np.mean(fold_aucs), auc_sd=np.std(fold_aucs),
        balanced_accuracy_mean=np.mean(fold_bas), balanced_accuracy_sd=np.std(fold_bas),
    )
    print(f"[CV-robust, {name}] AUC={np.mean(fold_aucs):.3f}+/-{np.std(fold_aucs):.3f}  "
          f"Balanced accuracy@0.5={np.mean(fold_bas):.3f}+/-{np.std(fold_bas):.3f}")

cv_robust_df = pd.DataFrame(cv_robust_results).T.sort_values('balanced_accuracy_mean', ascending=False)
print("\n=== Robust (50-fold repeated CV) model comparison -- this table drives model selection ===")
cv_robust_df.round(3)


## 18. Select the best model and evaluate on TEST — Table 3 equivalent

Selected programmatically by **50-fold repeated-CV balanced accuracy** (the table directly above), not the single validation split, among the five real classifiers (baselines are for comparison only, not eligible to be "the model"). This is evaluated on the test set **once**, per the original protocol; the decision threshold used is still the one tuned on the training fold above.

In [ ]:
best_model_name = cv_robust_df['balanced_accuracy_mean'].idxmax()
print(f"Best model by 50-fold repeated CV (balanced accuracy): {best_model_name}")

best_model = fitted_models[best_model_name]
best_build_fn = build_fns[best_model_name]
best_threshold = results[best_model_name]['threshold']

test_prob = best_model.predict_proba(X_test_scaled)[:, 1]
test_metrics, test_cm = eval_at(y_test, test_prob, best_threshold)
print_eval(f'{best_model_name} -- FINAL TEST EVALUATION', test_metrics, test_cm)


## 19. Robustness check — 100-fold repeated cross-validation

With only 44 real minority-class counties, a single split is noisy. Re-evaluates the winning model's hyperparameters across the whole real dataset. *(Generalizes CL's small-sample-noise point.)*

In [ ]:
X_full_imp = apply_impute(pd.concat([X_train, X_val, X_test]))
y_full = pd.concat([y_train, y_val, y_test])
X_full_scaled_for_cv = StandardScaler().fit_transform(X_full_imp[feature_cols])

rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=20, random_state=RANDOM_STATE)
aucs, bal_accs = [], []
for tr_idx, te_idx in rskf.split(X_full_scaled_for_cv, y_full):
    sw = np.where(y_full.values[tr_idx] == 0, (y_full.values[tr_idx] == 1).sum() / max((y_full.values[tr_idx] == 0).sum(), 1), 1.0)
    m = best_build_fn()
    try:
        m.fit(X_full_scaled_for_cv[tr_idx], y_full.values[tr_idx], sample_weight=sw)
    except TypeError:
        m.fit(X_full_scaled_for_cv[tr_idx], y_full.values[tr_idx])
    prob = m.predict_proba(X_full_scaled_for_cv[te_idx])[:, 1]
    aucs.append(roc_auc_score(y_full.values[te_idx], prob))
    bal_accs.append(balanced_accuracy_score(y_full.values[te_idx], (prob >= 0.5).astype(int)))
print(f"[Robustness: 100-fold repeated CV, {best_model_name}] AUC mean={np.mean(aucs):.3f} sd={np.std(aucs):.3f}")
print(f"[Robustness: 100-fold repeated CV, {best_model_name}] Balanced accuracy @0.5 mean={np.mean(bal_accs):.3f} sd={np.std(bal_accs):.3f}")


## 20. Leave-one-state-out (LOSO) check

Tests whether performance holds when an entire state is held out. *(Reviewer CL #6)*

In [ ]:
print("=== Leave-one-state-out balanced accuracy (geographic generalization check) ===")
X_full_states = pd.concat([X_train, X_val, X_test])['State']
for state in sorted(X_full_states.unique()):
    tr_mask = (X_full_states != state).values
    te_mask = (X_full_states == state).values
    if y_full.values[te_mask].sum() == 0 or y_full.values[te_mask].sum() == te_mask.sum():
        print(f"  {state}: skipped (only one class present in held-out state)")
        continue
    sw = np.where(y_full.values[tr_mask] == 0, (y_full.values[tr_mask] == 1).sum() / max((y_full.values[tr_mask] == 0).sum(), 1), 1.0)
    m = best_build_fn()
    try:
        m.fit(X_full_scaled_for_cv[tr_mask], y_full.values[tr_mask], sample_weight=sw)
    except TypeError:
        m.fit(X_full_scaled_for_cv[tr_mask], y_full.values[tr_mask])
    prob = m.predict_proba(X_full_scaled_for_cv[te_mask])[:, 1]
    ba = balanced_accuracy_score(y_full.values[te_mask], (prob >= 0.5).astype(int))
    print(f"  Held-out state = {state}: n={te_mask.sum()}, balanced accuracy={ba:.3f}")


## 21. SHAP — on real counties only, using the winning model — Table 5 / Figure 3 equivalent

The explainer type is chosen based on which model won: `TreeExplainer` for tree-based models (Decision Tree, Random Forest, XGBoost — fast and exact), `LinearExplainer` for Logistic Regression, or `KernelExplainer` as a model-agnostic fallback for SVM/KNN (slower, but matches what the original manuscript's Methods section described using). *(Reviewer CL #16)*

In [ ]:
import shap

def select_positive_class_shap(raw_shap_values):
    """Normalize a SHAP output for binary classification down to a single
    (n_samples, n_features) array of values for the positive class (label 1).
    Different shap versions return this differently:
      - older versions: a list of two (n_samples, n_features) arrays, one per class
      - newer versions (>=0.44 or so): a single (n_samples, n_features, n_classes) array
      - some explainers (e.g. LinearExplainer on a single-output model): already
        a plain (n_samples, n_features) array
    """
    if isinstance(raw_shap_values, list):
        return raw_shap_values[1]
    arr = np.array(raw_shap_values)
    if arr.ndim == 3:
        return arr[:, :, 1]
    return arr

X_real_all_scaled = scaler.transform(X_full_imp[feature_cols])  # real counties only, in train-fitted scale

if best_model_name in ('Decision Tree', 'Random Forest', 'XGBoost'):
    explainer = shap.TreeExplainer(best_model)
    shap_values = select_positive_class_shap(explainer.shap_values(X_real_all_scaled))
elif best_model_name == 'Logistic Regression':
    explainer = shap.LinearExplainer(best_model, X_train_scaled)
    shap_values = select_positive_class_shap(explainer.shap_values(X_real_all_scaled))
else:
    # Model-agnostic fallback (SVM / KNN) -- slower; subsample background for tractability
    background = shap.sample(X_train_scaled, min(100, len(X_train_scaled)), random_state=RANDOM_STATE)
    explainer = shap.KernelExplainer(best_model.predict_proba, background)
    shap_values = select_positive_class_shap(explainer.shap_values(X_real_all_scaled, nsamples=100))

print(f"SHAP explainer used: {type(explainer).__name__}")
print(f"SHAP values shape: {np.array(shap_values).shape}")

mean_abs_shap = np.abs(shap_values).mean(axis=0)
mean_shap_df = pd.DataFrame({'Feature': feature_cols, 'Mean Absolute SHAP Value': mean_abs_shap})
mean_shap_df['Weight'] = mean_shap_df['Mean Absolute SHAP Value'] / mean_shap_df['Mean Absolute SHAP Value'].sum()
mean_shap_df = mean_shap_df.sort_values('Mean Absolute SHAP Value', ascending=False).reset_index(drop=True)
print("\n=== SHAP mean |value| and weights (REAL counties only) -- Table 5 equivalent ===")
mean_shap_df.head(15)


In [ ]:
shap.summary_plot(shap_values, X_full_imp[feature_cols], feature_names=feature_cols)


## 22. PDI — recomputed on real counties only — Table 6/7, Figure 4 equivalent

Fixes a bug found in the original code: percentile ranks were previously computed against a reference population that was 47% synthetic SMOTE rows. Here, ranks are computed only among real counties.

**FIX (2026-09-20):** this cell previously determined each feature's direction (whether a high value pushes a county's PDI up or down) using `sign(mean signed SHAP value)`. That statistic is confounded by the dataset's 94%/6% class base rate and silently failed to flip about a third of the top-11 features' weight, so every county's PDI value was affected. The corrected method below uses the sign of the **Pearson correlation** between each feature's own value and its own SHAP value instead — the method already verified to reproduce the manuscript's actual published PDI results (top county: McCone, Montana, PDI 0.791).

In [ ]:
from scipy.stats import pearsonr

top_n = 11  # keep comparable to the original paper; adjust based on the SHAP ranking above if you prefer
top_features = mean_shap_df.head(top_n)['Feature'].tolist()
top_weights = mean_shap_df.head(top_n)['Weight'].values
top_weights = top_weights / top_weights.sum()  # renormalize over the subset actually used in the PDI

# --- FIX (2026-09-20): direction = sign of the Pearson correlation between each
# feature's own value and its own SHAP value -- NOT sign(mean signed SHAP), which is
# confounded by the 94%/6% class base rate. This matches shap_direction_v2.csv /
# pdi_results_corrected_v2.csv from an earlier session, which the manuscript's real,
# published Table 4/6/7 and Figure 4 are built from.
feature_corr = {}
for i, feat in enumerate(feature_cols):
    r, _ = pearsonr(X_real_all_scaled[:, i], shap_values[:, i])
    feature_corr[feat] = r
feature_direction = {feat: np.sign(r) for feat, r in feature_corr.items()}

shap_direction_df = pd.DataFrame({
    'Feature': feature_cols,
    'Feature-SHAP correlation': [feature_corr[f] for f in feature_cols],
    'Direction': ['Positive' if feature_direction[f] >= 0 else 'Negative' for f in feature_cols],
})

percentile_ranks = pd.DataFrame(index=X_full_imp.index)
for feat in top_features:
    vals = X_full_imp[feat].values
    ranks = rankdata(vals, method='average') / len(vals)
    if feature_direction[feat] < 0:
        ranks = 1 - ranks
    percentile_ranks[feat] = ranks

pdi_values = percentile_ranks[top_features].values.dot(top_weights)
pdi_df = pd.DataFrame({
    'County, State': df.loc[X_full_imp.index, 'County, State'].values,
    'PDI': np.round(pdi_values, 3)
}).sort_values('PDI', ascending=False).reset_index(drop=True)

print("=== PDI (recomputed, real counties only, CORRECTED direction method) -- Table 6/7 equivalent ===")
print("Top 10:")
print(pdi_df.head(10))
print("\nBottom 10:")
print(pdi_df.tail(10))
print("\nTier counts (0-0.4 / 0.4-0.6 / 0.6-1.0):")
print("  Low:", ((pdi_df['PDI'] >= 0) & (pdi_df['PDI'] <= 0.4)).sum())
print("  Medium:", ((pdi_df['PDI'] > 0.4) & (pdi_df['PDI'] <= 0.6)).sum())
print("  High:", ((pdi_df['PDI'] > 0.6) & (pdi_df['PDI'] <= 1.0)).sum())

# Per-feature breakdown for the #1 county -- this is the data the Principal Findings
# worked-example paragraph needs.
top_county = pdi_df.iloc[0]['County, State']
top_idx = df.index[df['County, State'] == top_county][0]
breakdown_rows = []
for feat, w in zip(top_features, top_weights):
    pct = percentile_ranks.loc[top_idx, feat]
    contrib = w * pct
    breakdown_rows.append({
        'Feature': feat,
        'Weight': round(float(w), 3),
        'Percentile Rank': round(float(pct), 3),
        'Contribution': round(float(contrib), 3),
    })
top_county_breakdown_df = pd.DataFrame(breakdown_rows).sort_values('Contribution', ascending=False)
print(f"\n=== Per-feature breakdown for top county: {top_county} (PDI = {pdi_df.iloc[0]['PDI']}) ===")
print(top_county_breakdown_df.to_string(index=False))
print(f"Sum of contributions: {top_county_breakdown_df['Contribution'].sum():.3f}  (should equal PDI = {pdi_df.iloc[0]['PDI']})")


In [ ]:
plt.figure(figsize=(10, 6))
pdi_df['PDI'].plot(kind='hist', bins=30, color='skyblue', edgecolor='white')
plt.title('Distribution of Pharmacy Desert Index (PDI) -- corrected, real counties only')
plt.xlabel('PDI')
plt.ylabel('Frequency')
plt.show()


## 23. Save results

In [ ]:
comparison_df.to_csv('model_comparison_corrected.csv')
cv_robust_df.to_csv('cv_robust_model_comparison.csv')
pd.DataFrame([test_metrics]).assign(model=best_model_name).to_csv('test_set_evaluation.csv', index=False)
pd.DataFrame({'auc': aucs, 'balanced_accuracy': bal_accs}).to_csv('robustness_100fold_cv.csv', index=False)
pdi_df.to_csv('pdi_results_corrected.csv', index=False)
mean_shap_df.to_csv('shap_weights_corrected.csv', index=False)
shap_direction_df.to_csv('shap_direction.csv', index=False)
top_county_breakdown_df.to_csv('top_county_breakdown.csv', index=False)
print("Saved: model_comparison_corrected.csv, cv_robust_model_comparison.csv, test_set_evaluation.csv, "
      "robustness_100fold_cv.csv, pdi_results_corrected.csv, shap_weights_corrected.csv, "
      "shap_direction.csv, top_county_breakdown.csv")
print(f"\nBest model (selected by 50-fold repeated CV): {best_model_name}")
print("\nDONE. Send back all 8 CSVs so the manuscript tables/figures can be cross-checked.")

from google.colab import files
files.download('model_comparison_corrected.csv')
files.download('cv_robust_model_comparison.csv')
files.download('test_set_evaluation.csv')
files.download('robustness_100fold_cv.csv')
files.download('pdi_results_corrected.csv')
files.download('shap_weights_corrected.csv')
files.download('shap_direction.csv')
files.download('top_county_breakdown.csv')


## 24. Export real per-county predicted probabilities (test set + out-of-fold CV)

Added 2026-09-20. Exports the fitted model's actual predicted probabilities so a genuine confusion matrix and calibration/Brier-score assessment can be computed outside this notebook (used to close reviewer items C2 and C4).

In [ ]:
from sklearn.metrics import brier_score_loss

test_prob_export = best_model.predict_proba(X_test_scaled)[:, 1]
test_pred_export = (test_prob_export >= best_threshold).astype(int)

xgboost_test_predictions_df = pd.DataFrame({
    'County': X_test_imp['County'].values,
    'State': X_test_imp['State'].values,
    'y_true': y_test.values,
    'predicted_probability': test_prob_export,
    'predicted_label': test_pred_export,
    'threshold_used': best_threshold,
})
xgboost_test_predictions_df.to_csv('xgboost_test_predictions.csv', index=False)

print("=== Sanity check: test set ===")
print("Brier score:", brier_score_loss(y_test, test_prob_export))
print("Confusion matrix [[TN,FP],[FN,TP]]:\n", confusion_matrix(y_test, test_pred_export))


In [ ]:
oof_rows = []
rskf_export = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=RANDOM_STATE)
for rep_fold_idx, (tr_idx, te_idx) in enumerate(rskf_export.split(X_trainval_scaled, y_trainval_arr)):
    repeat_num = rep_fold_idx // 5
    fold_num = rep_fold_idx % 5
    m = best_build_fn()
    try:
        m.fit(X_trainval_scaled[tr_idx], y_trainval_arr[tr_idx], sample_weight=sw_trainval[tr_idx])
    except TypeError:
        m.fit(X_trainval_scaled[tr_idx], y_trainval_arr[tr_idx])
    prob = m.predict_proba(X_trainval_scaled[te_idx])[:, 1]
    counties = X_trainval_imp['County'].values[te_idx]
    states = X_trainval_imp['State'].values[te_idx]
    for c, s, yt, p in zip(counties, states, y_trainval_arr[te_idx], prob):
        oof_rows.append({'repeat': repeat_num, 'fold': fold_num, 'County': c, 'State': s,
                          'y_true': yt, 'predicted_probability': p})

xgboost_cv_oof_predictions_df = pd.DataFrame(oof_rows)
xgboost_cv_oof_predictions_df.to_csv('xgboost_cv_oof_predictions.csv', index=False)

print(f"\n=== Sanity check: pooled out-of-fold CV ({len(xgboost_cv_oof_predictions_df)} rows) ===")
oof_pred_05 = (xgboost_cv_oof_predictions_df['predicted_probability'] >= 0.5).astype(int)
print("Pooled balanced accuracy @ 0.5:", balanced_accuracy_score(xgboost_cv_oof_predictions_df['y_true'], oof_pred_05))
print("Pooled Brier score:", brier_score_loss(xgboost_cv_oof_predictions_df['y_true'], xgboost_cv_oof_predictions_df['predicted_probability']))

from google.colab import files
files.download('xgboost_test_predictions.csv')
files.download('xgboost_cv_oof_predictions.csv')


## 26. FINAL REVIEW PACKAGE — all figures, CSVs, and numbers in one download

Added 2026-09-20. Run this LAST, after every cell above (in the same runtime, no restart). It regenerates every chart the notebook produces from the data/models already in memory (rather than assuming earlier figures are still open, since Colab's inline backend can close them after each cell), writes one text file with every key number (dataset sizes, all 3 model-comparison tables, test metrics, 100-fold robustness, leave-one-state-out balanced accuracy — recomputed here since it was previously only printed, never saved — SHAP direction/weights, and the full PDI table/tier counts/worked example), and bundles that together with every CSV already saved this run into one zip file.

In [ ]:
import os, shutil, zipfile
from datetime import datetime

REVIEW_DIR = 'review_package'
FIG_DIR = os.path.join(REVIEW_DIR, 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

summary_lines = []
def log(msg=''):
    print(msg)
    summary_lines.append(str(msg))

log('=' * 70)
log('PHARMACY DESERT INDEX -- FULL REVIEW PACKAGE')
log(f'Generated: {datetime.now().isoformat()}')
log('=' * 70)

log('\n--- Dataset ---')
log(f'Total real counties: {len(df)}')
log(f'Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}')
log('Train class balance:\n' + str(y_train.value_counts()))
log('Val class balance:\n' + str(y_val.value_counts()))
log('Test class balance:\n' + str(y_test.value_counts()))

log('\n--- Table 1 equivalent: 50-fold repeated CV model comparison ---')
log(cv_robust_df.round(4).to_string())

log('\n--- Validation model comparison (all models, incl. LR baseline) ---')
log(comparison_df.round(4).to_string())

log(f'\n--- Best model (by 50-fold CV): {best_model_name} ---')
log(f'Selected threshold: {best_threshold}')
log('Test-set evaluation:')
for k, v in test_metrics.items():
    log(f'  {k}: {v}')
log('Test confusion matrix [[TN,FP],[FN,TP]]:\n' + str(test_cm))

log('\n--- 100-fold repeated CV robustness ---')
log(f'AUC: mean={np.mean(aucs):.4f} sd={np.std(aucs):.4f} '
    f'2.5-97.5pct=[{np.percentile(aucs, 2.5):.4f}, {np.percentile(aucs, 97.5):.4f}]')
log(f'Balanced accuracy: mean={np.mean(bal_accs):.4f} sd={np.std(bal_accs):.4f} '
    f'2.5-97.5pct=[{np.percentile(bal_accs, 2.5):.4f}, {np.percentile(bal_accs, 97.5):.4f}]')

log('\n--- Leave-one-state-out balanced accuracy (recomputed here to actually save it) ---')
loso_rows = []
try:
    for state in sorted(X_full_states.unique()):
        tr_mask = (X_full_states != state).values
        te_mask = (X_full_states == state).values
        if y_full.values[te_mask].sum() == 0 or y_full.values[te_mask].sum() == te_mask.sum():
            log(f'  {state}: skipped (only one class present)')
            continue
        sw = np.where(y_full.values[tr_mask] == 0,
                      (y_full.values[tr_mask] == 1).sum() / max((y_full.values[tr_mask] == 0).sum(), 1), 1.0)
        m = best_build_fn()
        try:
            m.fit(X_full_scaled_for_cv[tr_mask], y_full.values[tr_mask], sample_weight=sw)
        except TypeError:
            m.fit(X_full_scaled_for_cv[tr_mask], y_full.values[tr_mask])
        prob = m.predict_proba(X_full_scaled_for_cv[te_mask])[:, 1]
        ba = balanced_accuracy_score(y_full.values[te_mask], (prob >= 0.5).astype(int))
        loso_rows.append({'State': state, 'n': int(te_mask.sum()), 'balanced_accuracy': ba})
        log(f'  {state}: n={te_mask.sum()}, balanced accuracy={ba:.4f}')
    loso_df = pd.DataFrame(loso_rows)
    loso_df.to_csv(os.path.join(REVIEW_DIR, 'loso_balanced_accuracy.csv'), index=False)
except Exception as e:
    log(f'Could not recompute LOSO: {e}')

log('\n--- SHAP weights, top 15 (Table 5 equivalent) ---')
log(mean_shap_df.head(15).round(4).to_string())
log('\n--- SHAP direction, top 11 features used in the PDI (Table 4 equivalent) ---')
log(shap_direction_df.set_index('Feature').loc[top_features].round(4).to_string())
log('Direction counts (all 46 features): ' + str(shap_direction_df['Direction'].value_counts().to_dict()))

log('\n--- PDI distribution (corrected) ---')
log(pdi_df['PDI'].describe().to_string())
log(f"Tier counts -- Low: {((pdi_df['PDI'] >= 0) & (pdi_df['PDI'] <= 0.4)).sum()}, "
    f"Medium: {((pdi_df['PDI'] > 0.4) & (pdi_df['PDI'] <= 0.6)).sum()}, "
    f"High: {((pdi_df['PDI'] > 0.6) & (pdi_df['PDI'] <= 1.0)).sum()}")
log('\nTop 15 counties by PDI:')
log(pdi_df.head(15).to_string(index=False))
log('\nBottom 15 counties by PDI:')
log(pdi_df.tail(15).to_string(index=False))
log(f'\nTop county worked example: {top_county} (PDI={pdi_df.iloc[0]["PDI"]})')
log(top_county_breakdown_df.to_string(index=False))

with open(os.path.join(REVIEW_DIR, 'review_summary.txt'), 'w') as f:
    f.write('\n'.join(summary_lines))

def save_current_fig(name):
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, name), dpi=150, bbox_inches='tight')
    plt.close()
    print('Saved figure:', name)

def save_feature_importance(name, importances, filename):
    try:
        s = pd.Series(importances, index=feature_cols).sort_values(ascending=False).head(20)
        plt.figure(figsize=(12, 6))
        s.plot(kind='bar', color='skyblue')
        plt.title(f'Feature Importances -- {name}')
        plt.ylabel('Importance')
        save_current_fig(filename)
    except Exception as e:
        print(f'Could not regenerate feature-importance plot for {name}:', e)

try:
    plt.figure(figsize=(10, max(6, len(feature_cols) * 0.25)))
    corr_with_target.sort_values().plot(kind='barh', color='steelblue')
    plt.title('Train-only correlation with Desert Y/N (Figure 2 equivalent)')
    plt.xlabel('Pearson correlation coefficient')
    save_current_fig('fig01_correlation_with_target.png')
except Exception as e:
    print('Could not regenerate fig01:', e)

try:
    plt.figure(figsize=(14, 11))
    sns.heatmap(train_corr_df.corr(), cmap='coolwarm', center=0, annot=False)
    plt.title('Correlation heatmap (training fold only)')
    save_current_fig('fig02_correlation_heatmap.png')
except Exception as e:
    print('Could not regenerate fig02:', e)

save_feature_importance('Logistic Regression', lr.coef_[0], 'fig03_feature_importance_logistic_regression.png')
save_feature_importance('Decision Tree', dt.feature_importances_, 'fig04_feature_importance_decision_tree.png')
save_feature_importance('Random Forest', rf.feature_importances_, 'fig05_feature_importance_random_forest.png')
save_feature_importance('XGBoost', xgb_model.feature_importances_, 'fig06_feature_importance_xgboost.png')
if svm_model.kernel == 'linear':
    save_feature_importance('SVM (linear)', svm_model.coef_[0], 'fig07_feature_importance_svm.png')

try:
    plt.figure(figsize=(10, 6))
    comparison_df['balanced_accuracy'].sort_values().plot(kind='barh', color='steelblue')
    plt.title('Model Comparison: Validation Balanced Accuracy')
    plt.xlabel('Balanced Accuracy')
    save_current_fig('fig08_model_comparison_balanced_accuracy.png')
except Exception as e:
    print('Could not regenerate fig08:', e)

try:
    plt.figure(figsize=(10, 6))
    comparison_df[['sensitivity', 'specificity']].plot(kind='bar')
    plt.title('Model Comparison: Sensitivity vs. Specificity (VAL)')
    plt.ylabel('Score')
    plt.xticks(rotation=45, ha='right')
    save_current_fig('fig09_model_comparison_sensitivity_specificity.png')
except Exception as e:
    print('Could not regenerate fig09:', e)

try:
    shap.summary_plot(shap_values, X_full_imp[feature_cols], feature_names=feature_cols, show=False)
    save_current_fig('fig10_shap_summary_plot.png')
except Exception as e:
    print('Could not regenerate fig10 (SHAP summary):', e)

try:
    plt.figure(figsize=(10, 6))
    pdi_df['PDI'].plot(kind='hist', bins=30, color='skyblue', edgecolor='white')
    plt.title('Distribution of Pharmacy Desert Index (PDI) -- corrected, real counties only')
    plt.xlabel('PDI')
    plt.ylabel('Frequency')
    save_current_fig('fig11_pdi_distribution_corrected.png')
except Exception as e:
    print('Could not regenerate fig11:', e)

csv_names = [
    'model_comparison_corrected.csv', 'cv_robust_model_comparison.csv', 'test_set_evaluation.csv',
    'robustness_100fold_cv.csv', 'pdi_results_corrected.csv', 'shap_weights_corrected.csv',
    'shap_direction.csv', 'top_county_breakdown.csv', 'xgboost_test_predictions.csv',
    'xgboost_cv_oof_predictions.csv',
]
for name in csv_names:
    if os.path.exists(name):
        shutil.copy(name, os.path.join(REVIEW_DIR, name))
    else:
        print(f'WARNING: {name} not found -- run the earlier save cells first.')

zip_path = 'pharmacy_desert_review_package.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, fnames in os.walk(REVIEW_DIR):
        for fname in fnames:
            full = os.path.join(root, fname)
            zf.write(full, os.path.relpath(full, REVIEW_DIR))

print('\nReview package built:', zip_path)
from google.colab import files
files.download(zip_path)
